# Notebook 07 — Iterative Burst Detector: Over-Kill Diagnostic

The iterative detector kills candidate bursts at four stages:

1. **Iteration trimming/merging** — candidates that fall below the composite threshold during refinement.
2. **Participation floor** — burstlets with peak active-unit fraction below `participation_floor` are dropped.
3. **BMI / LLR gate** — burstlets with `llr_aggregate < config.min_burst_modulation` (default `0.1`).
4. **GMM event clustering** — 6-component GMM on 6D event features, merged at distance ≤ `cluster_min_separation`, components scoring < 0 are discarded.

This notebook runs the detector on a real recording with `IterativeBurstTrace` enabled, then visualises each stage and projects the GMM feature space onto its first two PCA components so the inter-cluster geometry is visible in 2D.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA

repo_root = Path.cwd().resolve()
if (repo_root / 'pipeline_tasks').is_dir() is False and (repo_root.parent / 'pipeline_tasks').is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline_tasks.analysis import (
    IterativeBurstConfig,
    IterativeBurstTrace,
    compute_iterative_bursts,
)

REAL_DATA_ROOT = repo_root / 'data' / 'real_sorted_data'
REAL_KILOSORT_LABELS = {'good'}
print('repo_root:', repo_root)
print('REAL_DATA_ROOT exists:', REAL_DATA_ROOT.exists())

## Load a real recording

Loader helpers copied from notebook 06 so the two notebooks see the same data.

In [ ]:
def is_kilosort_dir(path: Path) -> bool:
    return path.is_dir() and all((path / fn).exists() for fn in ('spike_times.npy', 'spike_clusters.npy', 'params.py'))

def discover_real_spike_sources(root: Path | None) -> list[Path]:
    if root is None or not root.exists():
        return []
    if root.is_file() or is_kilosort_dir(root):
        return [root]
    curated = sorted(root.rglob('curated_spike_times.npy'))
    ks_dirs = sorted({p.parent for p in root.rglob('spike_times.npy') if is_kilosort_dir(p.parent)})
    curated_parents = {p.parent for p in curated}
    sources = curated + [p for p in ks_dirs if p not in curated_parents]
    return sorted(set(sources), key=lambda p: str(p))

def _read_kilosort_sample_rate(params_path: Path) -> float:
    values: dict[str, object] = {}
    exec(params_path.read_text(), {}, values)
    for key in ('sample_rate', 'fs', 'sampling_rate'):
        if key in values:
            return float(values[key])
    raise ValueError(f'Could not find sample_rate/fs/sampling_rate in {params_path}')

def _read_kilosort_keep_clusters(ks_dir: Path, labels: set[str] | None) -> set[int] | None:
    if labels is None:
        return None
    for filename in ('cluster_group.tsv', 'cluster_KSLabel.tsv'):
        label_path = ks_dir / filename
        if not label_path.exists():
            continue
        table = pd.read_csv(label_path, sep='\t')
        if 'cluster_id' not in table.columns:
            continue
        label_col = next((c for c in ('group', 'KSLabel', 'label') if c in table.columns), None)
        if label_col is None:
            continue
        keep = table[table[label_col].astype(str).str.lower().isin({l.lower() for l in labels})]
        return set(keep['cluster_id'].astype(int).tolist())
    return None

def load_kilosort_spike_times(ks_dir: Path, labels: set[str] | None = REAL_KILOSORT_LABELS) -> dict[str, np.ndarray]:
    ks_dir = Path(ks_dir)
    sample_rate = _read_kilosort_sample_rate(ks_dir / 'params.py')
    spike_samples = np.load(ks_dir / 'spike_times.npy').reshape(-1).astype(float)
    spike_clusters = np.load(ks_dir / 'spike_clusters.npy').reshape(-1).astype(int)
    keep_clusters = _read_kilosort_keep_clusters(ks_dir, labels)
    out: dict[str, np.ndarray] = {}
    for cid in np.unique(spike_clusters):
        if keep_clusters is not None and int(cid) not in keep_clusters:
            continue
        times = spike_samples[spike_clusters == cid] / sample_rate
        times = times[np.isfinite(times) & (times >= 0.0)]
        if times.size:
            out[f'cluster_{int(cid):03d}'] = np.sort(times.astype(float))
    if not out:
        raise ValueError(f'No clusters found in {ks_dir}')
    return out

sources = discover_real_spike_sources(REAL_DATA_ROOT)
print(f'discovered {len(sources)} real spike source(s):')
for s in sources:
    print('  ', s)
assert sources, f'no real recordings found under {REAL_DATA_ROOT}'

# Pick the first recording; change index to diagnose a different one.
SOURCE = sources[0]
RECORDING_NAME = SOURCE.name if SOURCE.is_dir() else SOURCE.parent.name
spike_times = load_kilosort_spike_times(SOURCE)
n_units = len(spike_times)
total_spikes = sum(s.size for s in spike_times.values())
t_min = min(float(s.min()) for s in spike_times.values() if s.size)
t_max = max(float(s.max()) for s in spike_times.values() if s.size)
print(f'recording: {RECORDING_NAME}')
print(f'units: {n_units}  spikes: {total_spikes}  duration: {t_max - t_min:.1f} s')

## Run the detector with a trace attached

In [ ]:
config = IterativeBurstConfig()
trace = IterativeBurstTrace()
result = compute_iterative_bursts(spike_times, config=config, debug=True, trace=trace)

print()
print(f'n_iterations: {len(trace.iterations)}  bin_size: {trace.bin_size*1000:.1f} ms')
print(f'final burstlets returned: {len(result.burstlets)}')
print(f'final network bursts:     {len(result.network_bursts)}')

## Kill-stage attribution at a glance

How many candidates survive each stage of the pipeline?

In [ ]:
n_seed_iter0 = trace.iterations[0]['n_candidates'] if trace.iterations else 0
n_converged = trace.iterations[-1]['n_candidates'] if trace.iterations else 0
n_pre_floor = len(trace.burstlets_pre_gates)
n_post_floor = trace.participation_gate['n_post'] if trace.participation_gate else n_pre_floor
n_pre_bmi = trace.bmi_gate['n_pre'] if trace.bmi_gate else n_post_floor
n_post_bmi = trace.bmi_gate['n_post'] if trace.bmi_gate else n_pre_bmi
if trace.gmm and 'kept_event_mask' in trace.gmm:
    n_post_gmm = int(trace.gmm['kept_event_mask'].sum())
    gmm_decision = trace.gmm.get('decision', 'unknown')
else:
    n_post_gmm = n_post_bmi
    gmm_decision = trace.gmm.get('skipped', 'unknown') if trace.gmm else 'no_trace'

attribution = pd.DataFrame([
    {'stage': '0. seed (iter 0)',         'n_candidates': n_seed_iter0, 'dropped_here': 0},
    {'stage': '1. converged candidates',  'n_candidates': n_converged,  'dropped_here': n_seed_iter0 - n_converged},
    {'stage': '2. pre-floor burstlets',   'n_candidates': n_pre_floor,  'dropped_here': max(0, n_converged - n_pre_floor)},
    {'stage': '3. after participation',   'n_candidates': n_post_floor, 'dropped_here': n_pre_floor - n_post_floor},
    {'stage': '4. after BMI/LLR',         'n_candidates': n_post_bmi,   'dropped_here': n_pre_bmi - n_post_bmi},
    {'stage': f'5. after GMM ({gmm_decision})', 'n_candidates': n_post_gmm, 'dropped_here': n_post_bmi - n_post_gmm},
])
attribution

## Stage 1 — iteration trimming and merging

How the composite signal and candidate count evolve across iterations. If a true burst is silently lost here, look for iterations where `n_candidates` drops sharply or the composite threshold climbs above the burst peak.

In [ ]:
t = trace.t_centers
iters = trace.iterations
show_iters = [0, len(iters) // 2, len(iters) - 1] if len(iters) >= 2 else [len(iters) - 1]
show_iters = sorted(set(i for i in show_iters if 0 <= i < len(iters)))

fig, axes = plt.subplots(len(show_iters) + 1, 1, figsize=(12, 2.2 * (len(show_iters) + 1)), sharex=False)
if len(show_iters) + 1 == 1:
    axes = [axes]

for ax, k in zip(axes[:-1], show_iters):
    snap = iters[k]
    ax.plot(t, snap['composite'], lw=0.6, color='steelblue')
    ax.axhline(snap['composite_threshold'], color='red', lw=0.8, ls='--', label=f"thr={snap['composite_threshold']:.2f}")
    ax.axhline(snap['composite_baseline'], color='gray', lw=0.6, ls=':', label=f"base={snap['composite_baseline']:.2f}")
    for c in snap['candidates']:
        ax.axvspan(c['start'], c['end'], color='orange', alpha=0.15)
    ax.set_title(f"iter {snap['iter']}  n_candidates={snap['n_candidates']}  delta={snap['convergence_delta']:.4f}")
    ax.set_ylabel('composite')
    ax.legend(loc='upper right', fontsize=8)

ax_count = axes[-1]
ax_count.plot([s['iter'] for s in iters], [s['n_candidates'] for s in iters], marker='o', color='black')
ax_count.set_xlabel('iteration')
ax_count.set_ylabel('n_candidates')
ax_count.set_title('candidate count per iteration')
ax_count.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Stage 2 — participation floor

Each converged candidate becomes a burstlet only if its peak active-unit fraction (`peak_synchrony`) clears the floor. With few units the floor is generous (`max(5, 0.15·n_units)/n_units`); with many units it tightens to `max(10, 0.05·n_units)/n_units`. Events to the **left** of the red line are killed at this stage.

In [ ]:
pre_floor = pd.DataFrame(trace.burstlets_pre_gates)
gate = trace.participation_gate or {}
floor = float(gate.get('floor', 0.0))
if pre_floor.empty:
    print('no pre-floor events')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    kept_mask = pre_floor['peak_synchrony'] >= floor
    axes[0].hist(pre_floor.loc[kept_mask, 'peak_synchrony'], bins=30, color='steelblue', alpha=0.8, label=f'kept ({kept_mask.sum()})')
    axes[0].hist(pre_floor.loc[~kept_mask, 'peak_synchrony'], bins=30, color='crimson', alpha=0.6, label=f'killed ({(~kept_mask).sum()})')
    axes[0].axvline(floor, color='red', ls='--', lw=1.2, label=f'floor={floor:.3f}')
    axes[0].set_xlabel('peak_synchrony'); axes[0].set_ylabel('count')
    axes[0].set_title('participation floor — peak_synchrony distribution')
    axes[0].legend()

    axes[1].scatter(pre_floor['peak_synchrony'], pre_floor['duration_s'], c=np.where(kept_mask, 'steelblue', 'crimson'), s=14, alpha=0.7)
    axes[1].axvline(floor, color='red', ls='--', lw=1.2)
    axes[1].set_xlabel('peak_synchrony'); axes[1].set_ylabel('duration_s')
    axes[1].set_title('killed (red) vs kept (blue)')
    plt.tight_layout(); plt.show()
    print(f"dropped at participation floor: {gate.get('n_dropped', 0)} / {gate.get('n_pre', len(pre_floor))}")

## Stage 3 — BMI / LLR gate

After the participation floor, each surviving burstlet is checked against `min_burst_modulation = config.min_burst_modulation` on its `llr_aggregate` column. Set to `0` to disable the gate entirely.

In [ ]:
bmi = trace.bmi_gate or {}
thr = float(bmi.get('threshold', 0.0))
pre_bmi = pd.DataFrame(bmi.get('pre_events', []))
if pre_bmi.empty:
    print('no events reached the BMI gate')
else:
    kept_bmi = pre_bmi['llr_aggregate'] >= thr
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].hist(pre_bmi.loc[kept_bmi, 'llr_aggregate'], bins=30, color='steelblue', alpha=0.8, label=f'kept ({kept_bmi.sum()})')
    axes[0].hist(pre_bmi.loc[~kept_bmi, 'llr_aggregate'], bins=30, color='crimson', alpha=0.6, label=f'killed ({(~kept_bmi).sum()})')
    axes[0].axvline(thr, color='red', ls='--', lw=1.2, label=f'threshold={thr:.2f}')
    axes[0].set_xlabel('llr_aggregate'); axes[0].set_ylabel('count')
    axes[0].set_title(f"BMI gate (enabled={bmi.get('enabled', False)})")
    axes[0].legend()

    axes[1].scatter(pre_bmi['llr_aggregate'], pre_bmi['participation'], c=np.where(kept_bmi, 'steelblue', 'crimson'), s=14, alpha=0.7)
    axes[1].axvline(thr, color='red', ls='--', lw=1.2)
    axes[1].set_xlabel('llr_aggregate'); axes[1].set_ylabel('participation')
    axes[1].set_title('killed (red) vs kept (blue)')
    plt.tight_layout(); plt.show()
    print(f"BMI gate kept {bmi.get('n_post', 0)} / {bmi.get('n_pre', 0)} at threshold {thr:.2f}")

## Stage 4 — GMM clustering (primary focus)

A 6-component GMM is fit on standardised 6D event features  
`[composite_peak, composite_mean, llr_aggregate, ff_peak, participation, burst_peak]`,  
components within `cluster_min_separation = 1.5` (standardised Euclidean) are merged, and merged clusters are scored

$$\text{score} = 0.35\,\text{composite\_peak} + 0.15\,\text{composite\_mean} + 0.20\,\text{llr\_aggregate} + 0.10\,\text{ff\_peak} + 0.15\,\text{participation} + 0.05\,\text{burst\_peak}$$

Clusters scoring `< 0` are discarded. PCA projects the 6D space to 2D so the inter-cluster geometry — and which cluster is sitting on the kill line — is visible.

In [ ]:
gmm = trace.gmm
if not gmm or 'X' not in gmm:
    reason = gmm.get('skipped') if gmm else 'no trace'
    print(f'GMM stage did not run — reason: {reason}')
else:
    X = gmm['X']
    Xs = gmm['X_scaled']
    labels = gmm['labels']
    centers_scaled = gmm['component_means_scaled']
    merged_groups = gmm['merged_groups']
    cluster_scores = np.asarray(gmm['cluster_scores'])
    kept = gmm['kept_event_mask']
    feature_names = gmm['feature_names']

    pca = PCA(n_components=2)
    Z = pca.fit_transform(Xs)
    Zc = pca.transform(centers_scaled)
    ev = pca.explained_variance_ratio_

    # Build a per-event "merged group id" array using the GMM component → group map
    comp_to_group = {}
    for gi, g in enumerate(merged_groups):
        for m in g['members']:
            comp_to_group[int(m)] = gi
    group_id = np.array([comp_to_group.get(int(lb), -1) for lb in labels])

    # Per-event score = score of its merged group
    per_event_score = np.array([cluster_scores[g] if g >= 0 else np.nan for g in group_id])

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))

    # (a) initial GMM components
    cmap1 = plt.get_cmap('tab10')
    for k in np.unique(labels):
        m = labels == k
        axes[0, 0].scatter(Z[m, 0], Z[m, 1], s=22, alpha=0.7, color=cmap1(k % 10), label=f'comp {k}')
    axes[0, 0].scatter(Zc[:, 0], Zc[:, 1], marker='X', s=140, c='black', edgecolor='white', lw=1.2)
    axes[0, 0].set_title(f'(a) initial GMM components (n={len(centers_scaled)})')
    axes[0, 0].legend(fontsize=7, ncol=2)

    # (b) merged groups
    cmap2 = plt.get_cmap('Set1')
    for g in np.unique(group_id):
        m = group_id == g
        axes[0, 1].scatter(Z[m, 0], Z[m, 1], s=22, alpha=0.7, color=cmap2(g % 9), label=f'group {g} (score={cluster_scores[g]:+.2f})')
    # group centroids in PC space — average of component means in that group
    for gi, g in enumerate(merged_groups):
        members = list(g['members'])
        cg = pca.transform(centers_scaled[members].mean(axis=0, keepdims=True))[0]
        axes[0, 1].scatter(cg[0], cg[1], marker='X', s=160, c='black', edgecolor='white', lw=1.4)
    axes[0, 1].set_title(f"(b) merged groups (n={len(merged_groups)}) — decision: {gmm.get('decision', '?')}")
    axes[0, 1].legend(fontsize=7)

    # (c) kept vs killed
    axes[1, 0].scatter(Z[kept, 0], Z[kept, 1], s=24, color='steelblue', alpha=0.8, label=f'kept ({int(kept.sum())})')
    axes[1, 0].scatter(Z[~kept, 0], Z[~kept, 1], s=24, color='crimson', alpha=0.8, label=f'killed ({int((~kept).sum())})')
    axes[1, 0].set_title('(c) kept vs killed')
    axes[1, 0].legend()

    # (d) cluster score as continuous color
    sc = axes[1, 1].scatter(Z[:, 0], Z[:, 1], c=per_event_score, cmap='RdBu_r', s=26, edgecolor='k', lw=0.2, vmin=-abs(cluster_scores).max(), vmax=abs(cluster_scores).max())
    plt.colorbar(sc, ax=axes[1, 1], label='cluster score (kill at 0)')
    axes[1, 1].set_title('(d) per-event cluster score — red bins are killed')

    for ax in axes.flat:
        ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}% var)')
        ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}% var)')
        ax.grid(alpha=0.2)
    fig.suptitle(f'GMM 2D PCA — {RECORDING_NAME}  (explained var: PC1+PC2 = {ev.sum()*100:.1f}%)')
    plt.tight_layout(); plt.show()

In [ ]:
# Pairwise inter-centroid distance table vs cluster_min_separation
if trace.gmm and 'merged_groups' in trace.gmm:
    centroids = np.stack([np.asarray(g['centroid']) for g in trace.gmm['merged_groups']])
    n_g = centroids.shape[0]
    D = np.zeros((n_g, n_g))
    for i in range(n_g):
        for j in range(n_g):
            D[i, j] = np.linalg.norm(centroids[i] - centroids[j])
    dist_df = pd.DataFrame(D, index=[f'g{i}' for i in range(n_g)], columns=[f'g{i}' for i in range(n_g)])
    print(f'cluster_min_separation = {config.cluster_min_separation}  (groups closer than this would have been merged)')
    display(dist_df.round(2))
else:
    print('no merged groups to compare')

In [ ]:
# Per-feature centroids: which features push each group below the kill line?
if trace.gmm and 'merged_groups' in trace.gmm:
    feature_names = trace.gmm['feature_names']
    weights = trace.gmm['score_weights']
    centroids = np.stack([np.asarray(g['centroid']) for g in trace.gmm['merged_groups']])
    scores = trace.gmm['cluster_scores']
    fig, ax = plt.subplots(figsize=(11, 4.5))
    x = np.arange(len(feature_names))
    width = 0.8 / max(1, centroids.shape[0])
    cmap = plt.get_cmap('Set1')
    for gi, c in enumerate(centroids):
        ax.bar(x + gi * width, c, width=width, color=cmap(gi % 9), label=f'group {gi}  score={scores[gi]:+.2f}')
    ax.set_xticks(x + width * (centroids.shape[0] - 1) / 2)
    ax.set_xticklabels([f'{fn}\n(w={w:.2f})' for fn, w in zip(feature_names, weights)], fontsize=9)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_ylabel('standardised centroid value')
    ax.set_title('Per-feature centroids by merged group (score = weighted sum across features)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2, axis='y')
    plt.tight_layout(); plt.show()

## Cross-stage flow

How many candidates each kill stage drops, side by side. The tallest red bar is the over-killer.

In [ ]:
stages = ['iter trim', 'participation', 'BMI/LLR', 'GMM']
dropped = [
    max(0, n_converged - n_pre_floor),
    n_pre_floor - n_post_floor,
    n_pre_bmi - n_post_bmi,
    max(0, n_post_bmi - n_post_gmm),
]
survivors_after = [n_pre_floor, n_post_floor, n_post_bmi, n_post_gmm]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(stages, survivors_after, color='steelblue', label='survivors after stage')
ax.bar(stages, dropped, bottom=survivors_after, color='crimson', label='killed at stage')
for i, (s, d) in enumerate(zip(survivors_after, dropped)):
    ax.text(i, s + d + 0.5, f'kill={d}', ha='center', fontsize=9)
ax.set_ylabel('events')
ax.set_title(f'kill-stage attribution — {RECORDING_NAME}')
ax.legend()
plt.tight_layout(); plt.show()

## Notes / conclusions

_Fill in after inspecting the plots above._

- Which stage drops the most candidates on this recording?
- In the GMM PCA panels, do the killed (crimson) events cluster tightly, or are they smeared along the kept side?
- Which feature(s) push the killed merged group below the `score = 0` line?
- Would lowering `min_burst_modulation`, the GMM `cluster_min_separation`, or the kill score threshold recover the lost candidates?

## Per-bin PCA across LDA iterations — `cx138_44_02` deep dive

The cells above operate on the **event-level** GMM (one row per detected event).
That view is downstream of the LDA refinement loop, so it cannot show why a
particular bin was assigned to the burst class in the first place.

`cx138_44_02` is heterogeneous: about 10 bursts at the end and silence at the
start. With the default config, the BMI/LLR gate deletes every burstlet; with
the gate disabled, the algorithm wraps the silent stretch into one giant
"burst". Both failures point to the **per-bin two-class partition** as the
real culprit.

The next cells:

1. reload the recording as `cx138_44_02`,
2. run the detector twice (default vs `min_burst_modulation=0`),
3. project `X_norm` (the feature matrix the Fisher LDA actually fits) to 2D
   PCA at every iteration so the boundary drift is visible,
4. fit `GaussianMixture` with `k ∈ {2,3,4,5}` on the **converged-iteration**
   bin features to test whether the data really wants more than two classes.


In [ ]:
# A — Override SOURCE to cx138_44_02 and reload spikes

# cx138_36_10
# cx138_36_11
# cx138_44_02

target = next((s for s in sources if 'cx138_36_10' in str(s)), None)
assert target is not None, f"file not found under {REAL_DATA_ROOT}"
SOURCE = target
RECORDING_NAME = SOURCE.name if SOURCE.is_dir() else SOURCE.parent.name
spike_times = load_kilosort_spike_times(SOURCE)
n_units = len(spike_times)
total_spikes = sum(s.size for s in spike_times.values())
t_min = min(float(s.min()) for s in spike_times.values() if s.size)
t_max = max(float(s.max()) for s in spike_times.values() if s.size)
print(f'recording: {RECORDING_NAME}')
print(f'units: {n_units}  spikes: {total_spikes}  duration: {t_max - t_min:.1f} s')


### B — Two reference runs

`trace_default` = stock config (BMI gate at 0.1, GMM on) — the configuration
that kills every burstlet.
`trace_no_gate` = `min_burst_modulation=0` and `cluster_events=False` — the
configuration that keeps everything and over-merges silence into a long burst.


In [ ]:
# B — Run the detector twice with traces attached
config_default = IterativeBurstConfig()
trace_default = IterativeBurstTrace()
result_default = compute_iterative_bursts(
    spike_times, config=config_default, debug=True, trace=trace_default,
)

print()
print('---  no-gate run  ---')
config_no_gate = IterativeBurstConfig(min_burst_modulation=0.0, cluster_events=False)
trace_no_gate = IterativeBurstTrace()
result_no_gate = compute_iterative_bursts(
    spike_times, config=config_no_gate, debug=True, trace=trace_no_gate,
)

print()
print(f'default      : burstlets={len(result_default.burstlets)}  net_bursts={len(result_default.network_bursts)}')
print(f'no-gate run  : burstlets={len(result_no_gate.burstlets)}  net_bursts={len(result_no_gate.network_bursts)}')

print()
print(f'trace_default iterations : {len(trace_default.iterations)}')
print(f'trace_no_gate iterations : {len(trace_no_gate.iterations)}')
print(f'feature_names            : {trace_default.feature_names}')
print(f'X_norm shape (iter 0)    : {trace_default.iterations[0]["X_norm"].shape}')


### C — Per-iteration 2D PCA grid

For each shown iteration, fit `PCA(n_components=2)` on that iteration's
`X_norm` and scatter all bins colored by the burst label **output by that
iteration's LDA**. The orange arrow is the Fisher direction `w` projected into
PC space — it points along the axis the LDA is using to separate classes.

Per-iteration PCA fit is intentional: the z-norm changes each iteration, so
a fixed PCA basis would not reflect what the LDA actually saw at step k.

Flip `TRACE_TO_PLOT` to inspect the other run, and set `PLOT_ALL_ITERS=True`
for the full per-iteration sweep.


In [ ]:
# C — Per-iteration PCA grid (output labels)
from sklearn.decomposition import PCA

TRACE_TO_PLOT = trace_default        # flip to trace_no_gate to see the other failure mode
PLOT_ALL_ITERS = False                # True → one panel per iteration

iters = TRACE_TO_PLOT.iterations
if PLOT_ALL_ITERS:
    show_iters = list(range(len(iters)))
else:
    show_iters = sorted({0, len(iters) // 2, len(iters) - 1})

feature_names = TRACE_TO_PLOT.feature_names

n_panels = len(show_iters)
n_cols = min(3, n_panels)
n_rows = (n_panels + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.4 * n_rows), squeeze=False)
axes_flat = axes.flat

for ax, k in zip(axes_flat, show_iters):
    snap = iters[k]
    Xn = snap['X_norm']
    labels = snap['candidate_mask'].astype(bool)
    w = snap['w']

    pca = PCA(n_components=2).fit(Xn)
    Z = pca.transform(Xn)
    w_pc = pca.components_ @ w
    arrow_scale = float(np.abs(Z).max()) * 0.6 / max(np.linalg.norm(w_pc), 1e-9)

    ax.scatter(Z[~labels, 0], Z[~labels, 1], s=4, alpha=0.35, color='steelblue', label=f'bg ({int((~labels).sum())})')
    ax.scatter(Z[labels, 0],  Z[labels, 1],  s=6, alpha=0.65, color='crimson',  label=f'burst ({int(labels.sum())})')
    ax.arrow(0, 0, w_pc[0] * arrow_scale, w_pc[1] * arrow_scale,
             color='darkorange', width=0.02, head_width=0.18, length_includes_head=True, alpha=0.9)
    top3 = sorted(zip(feature_names, w), key=lambda x: -abs(x[1]))[:3]
    wstr = ', '.join(f'{n}={v:+.2f}' for n, v in top3)
    ev = pca.explained_variance_ratio_
    ax.set_title(
        f"iter {snap['iter']}  n_cand={snap['n_candidates']}  "
        f"thr={snap['composite_threshold']:.2f}  Δ={snap['convergence_delta']:.3f}\n"
        f"PC1={ev[0]*100:.1f}%  PC2={ev[1]*100:.1f}%   top w: {wstr}",
        fontsize=9,
    )
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(alpha=0.2)
    ax.legend(loc='upper right', fontsize=7)

for ax in list(axes_flat)[n_panels:]:
    ax.axis('off')

fig.suptitle(
    f'Per-bin PCA per LDA iteration — {RECORDING_NAME}  '
    f'({"default" if TRACE_TO_PLOT is trace_default else "no-gate"})',
    fontsize=11,
)
plt.tight_layout(); plt.show()


### D — Boundary shift per iteration (input vs output labels)

Same PCA basis as section C, side by side: left = bins **fed to** this
iteration's Fisher fit (`candidate_mask_in`); right = bins **labelled by**
this iteration's composite (`candidate_mask`). The differences expose how
aggressively each iteration moves the boundary.


In [ ]:
# D — Input vs output labels per iteration
iters = TRACE_TO_PLOT.iterations
show_iters_io = sorted({0, len(iters) // 2, len(iters) - 1})

fig, axes = plt.subplots(len(show_iters_io), 2, figsize=(11, 4.0 * len(show_iters_io)), squeeze=False)

for row, k in enumerate(show_iters_io):
    snap = iters[k]
    Xn = snap['X_norm']
    li = snap['candidate_mask_in'].astype(bool)
    lo = snap['candidate_mask'].astype(bool)

    pca = PCA(n_components=2).fit(Xn)
    Z = pca.transform(Xn)
    for col, (mask, name) in enumerate([(li, 'input'), (lo, 'output')]):
        ax = axes[row, col]
        ax.scatter(Z[~mask, 0], Z[~mask, 1], s=4, alpha=0.35, color='steelblue')
        ax.scatter(Z[mask,  0], Z[mask,  1], s=6, alpha=0.65, color='crimson')
        ax.set_title(f"iter {snap['iter']} — {name} labels   burst={int(mask.sum())} / {mask.size}", fontsize=9)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        ax.grid(alpha=0.2)

fig.suptitle(
    f'Iteration boundary shift — {RECORDING_NAME}  '
    f'({"default" if TRACE_TO_PLOT is trace_default else "no-gate"})',
    fontsize=11,
)
plt.tight_layout(); plt.show()


### E — 3D PCA at the converged iteration

Sanity check: in 3D, do the burst and background bins form two clean blobs,
or is there visible substructure (e.g. silence vs random firing) that the
LDA is folding into a single "background" class?


In [ ]:
# E — 3D PCA at the converged iteration
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers projection)

snap_f = TRACE_TO_PLOT.iterations[-1]
Xn = snap_f['X_norm']
labels = snap_f['candidate_mask'].astype(bool)

pca3 = PCA(n_components=3).fit(Xn)
Z3 = pca3.transform(Xn)

fig = plt.figure(figsize=(8, 6.5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(Z3[~labels, 0], Z3[~labels, 1], Z3[~labels, 2], s=4, alpha=0.3, color='steelblue', label=f'bg ({int((~labels).sum())})')
ax.scatter(Z3[labels,  0], Z3[labels,  1], Z3[labels,  2], s=8, alpha=0.7, color='crimson',  label=f'burst ({int(labels.sum())})')
ev = pca3.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}%)')
ax.set_zlabel(f'PC3 ({ev[2]*100:.1f}%)')
ax.set_title(f'Converged-iteration 3D PCA — {RECORDING_NAME}')
ax.legend()
plt.tight_layout(); plt.show()


### F — Multi-k GMM on bin features (4-cluster hypothesis)

On the **converged-iteration** `X_norm`, fit `GaussianMixture` with
`n_components ∈ {2, 3, 4, 5}`. BIC is printed for each k; the k that BIC
prefers tells you how many latent components the data really wants. The
panels project every fit into the same converged-PCA basis used in section E
so you can compare clusterings directly.

Reasoning: if BIC strongly prefers `k ≥ 3` and the cluster centroids in the
**original feature space** map to interpretable regimes (silence, random
firing, build-up, true burst), the Fisher LDA's 2-class assumption is the
root cause of the bad behavior on `cx138_44_02`.


In [ ]:
# F — GaussianMixture k-sweep + BIC on bin features
from sklearn.mixture import GaussianMixture

snap_f = TRACE_TO_PLOT.iterations[-1]
Xn = snap_f['X_norm']
feature_names = TRACE_TO_PLOT.feature_names

pca2 = PCA(n_components=2).fit(Xn)
Z2 = pca2.transform(Xn)
ev = pca2.explained_variance_ratio_

ks = [2, 3, 4, 5]
gmm_fits = {}
bic_rows = []
for k in ks:
    gm = GaussianMixture(n_components=k, n_init=5, random_state=42, reg_covar=1e-6).fit(Xn)
    cl = gm.predict(Xn)
    bic_rows.append({'k': k, 'BIC': float(gm.bic(Xn)), 'AIC': float(gm.aic(Xn))})
    gmm_fits[k] = (gm, cl)

bic_df = pd.DataFrame(bic_rows).set_index('k')
print('BIC / AIC by k (lower is better):')
display(bic_df.round(1))
best_k = int(bic_df['BIC'].idxmin())
print(f'\nBIC-preferred k = {best_k}')

fig, axes = plt.subplots(1, len(ks), figsize=(4.4 * len(ks), 4.4), squeeze=False)
cmap = plt.get_cmap('tab10')
for ax, k in zip(axes[0], ks):
    gm, cl = gmm_fits[k]
    for c in range(k):
        m = cl == c
        ax.scatter(Z2[m, 0], Z2[m, 1], s=5, alpha=0.5, color=cmap(c % 10), label=f'c{c} ({m.sum()})')
    ax.set_title(f'k={k}  BIC={bic_df.loc[k, "BIC"]:.0f}')
    ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}%)')
    ax.grid(alpha=0.2)
    ax.legend(fontsize=6, loc='upper right')

fig.suptitle(f'Bin-level GMM k-sweep on converged X_norm — {RECORDING_NAME}', fontsize=11)
plt.tight_layout(); plt.show()

# Centroids in the original (z-scored) feature space — interpret each cluster
print(f'\nCluster centroids in standardised feature space (k={best_k}):')
gm_best, cl_best = gmm_fits[best_k]
centroids = pd.DataFrame(gm_best.means_, columns=feature_names)
centroids.insert(0, 'time_share', [float((cl_best == c).mean()) for c in range(best_k)])
display(centroids.round(2))


### G — Cluster assignment as a time strip

Project the BIC-preferred k assignment back onto the recording timeline. The
top strip is the per-bin cluster id; below it are the participation and
composite signals so you can visually verify whether a cluster e.g. covers
the late-recording bursts, the silent stretch, or something in between.


In [ ]:
# G — Time strip of bin-level cluster assignment
t = TRACE_TO_PLOT.t_centers
snap_f = TRACE_TO_PLOT.iterations[-1]
composite_final = snap_f['composite']
labels_final = snap_f['candidate_mask'].astype(bool)

# Use the BIC-preferred k from section F
gm_best, cl_best = gmm_fits[best_k]
strip = cl_best.reshape(1, -1)

participation = result_default.plot_data['participation_signal'] if TRACE_TO_PLOT is trace_default else result_no_gate.plot_data['participation_signal']

fig, axes = plt.subplots(3, 1, figsize=(13, 6), sharex=True, gridspec_kw={'height_ratios': [0.5, 1, 1]})

axes[0].imshow(strip, aspect='auto', cmap='tab10', interpolation='nearest',
               extent=(float(t[0]), float(t[-1]), 0, 1), vmin=0, vmax=9)
axes[0].set_yticks([])
axes[0].set_title(f'k={best_k} bin clusters over time — {RECORDING_NAME}')

axes[1].plot(t, participation, lw=0.6, color='steelblue')
axes[1].set_ylabel('participation')
axes[1].grid(alpha=0.2)

axes[2].plot(t, composite_final, lw=0.6, color='black')
axes[2].axhline(snap_f['composite_threshold'], color='red', ls='--', lw=0.7,
                label=f"thr={snap_f['composite_threshold']:.2f}")
axes[2].fill_between(t, composite_final.min(), composite_final.max(),
                     where=labels_final, color='crimson', alpha=0.12, label='LDA burst')
axes[2].set_ylabel('composite')
axes[2].set_xlabel('time (s)')
axes[2].legend(loc='upper right', fontsize=8)
axes[2].grid(alpha=0.2)

plt.tight_layout(); plt.show()


### Notes — what to look for on `cx138_44_02`

- **Section C / D (per-iter PCA):** does the LDA's burst class (red) actually
  occupy a coherent region of PC space, or is it stretched across a
  silence-only stripe? If the latter, the LDA centroid is being dragged by
  the silent block.
- **Section E (3D):** if the burst points form a small cluster well-separated
  from the rest but the rest of the bins look like ≥ 2 sub-blobs, the
  2-class assumption is wrong for this recording.
- **Section F (BIC):** if BIC prefers `k ≥ 3` by a large margin, the per-bin
  feature space wants a multi-class model. Look at the centroid table to
  name the regimes (high-P high-LLR = burst, near-zero = silence, high-FF
  low-P = random firing, etc.).
- **Section G (time strip):** confirms whether the BIC-preferred clustering
  matches the visible structure of the recording (early silence, late
  bursts, build-up in between).

What this section deliberately does **not** do: change the detector. Use the
geometry above to decide the next change (e.g. replace the inner LDA with a
GMM-EM step, raise the seed percentile, gate on cluster identity rather than
a fixed LLR threshold).
